In [3]:
from pyspark.sql import functions as F

df_silver_orders = (
    spark.read
        .format("delta")
        .load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Silver.Lakehouse/Tables/dbo/fact_seo_orders")
)

print("Silver orders rows:", df_silver_orders.count())
print("Silver orders columns:", len(df_silver_orders.columns))

display(df_silver_orders.limit(10))

StatementMeta(, c28aaa3f-cd92-4fb3-b83d-6c7b339447cf, 5, Finished, Available, Finished, False)

Silver orders rows: 4987
Silver orders columns: 9


SynapseWidget(Synapse.DataFrame, e36c1ad2-b3ca-4bf1-a9d3-f87828138382)

In [2]:
df_date = spark.read.table("dim_date")
df_country = spark.read.table("dim_country")
df_product = spark.read.table("dim_product")
df_page = spark.read.table("dim_page")
df_keyword = spark.read.table("dim_keyword")

StatementMeta(, 08e16308-3a79-4a50-b2fe-37d08b77cc99, 4, Finished, Available, Finished, False)

In [4]:
from pyspark.sql import functions as F

df_date = spark.read.table("dim_date")
df_country = spark.read.table("dim_country")
df_product = spark.read.table("dim_product")
df_page = spark.read.table("dim_page")
df_keyword = spark.read.table("dim_keyword")

print("Date:", df_date.count())
print("Country:", df_country.count())
print("Product:", df_product.count())
print("Page:", df_page.count())
print("Keyword:", df_keyword.count())

StatementMeta(, c28aaa3f-cd92-4fb3-b83d-6c7b339447cf, 6, Finished, Available, Finished, False)

Date: 365
Country: 10
Product: 50
Page: 500
Keyword: 2000


In [5]:
df_gold_orders = (
    df_silver_orders

    # Date
    .join(
        df_date.select("date", "date_key"),
        df_silver_orders.order_date == df_date.date,
        "left"
    )

    # Country
    .join(
        df_country.select("country_code", "country_key"),
        df_silver_orders.country_code == df_country.country_code,
        "left"
    )

    # Product
    .join(
        df_product.select("product_id", "product_key"),
        df_silver_orders.product_id == df_product.product_id,
        "left"
    )

    # Page
    .join(
        df_page.select("page_id", "page_key"),
        df_silver_orders.page_id == df_page.page_id,
        "left"
    )

    # Keyword
    .join(
        df_keyword.select("keyword_id", "keyword_key"),
        df_silver_orders.keyword_id == df_keyword.keyword_id,
        "left"
    )

    .select(
        df_silver_orders.order_id,
        "date_key",
        "country_key",
        "product_key",
        "page_key",
        "keyword_key",
        df_silver_orders.order_amount,
        df_silver_orders.order_status,
        df_silver_orders.updated_timestamp
    )
)

print("Gold orders rows:", df_gold_orders.count())
print("Gold orders columns:", len(df_gold_orders.columns))

display(df_gold_orders.limit(10))

StatementMeta(, c28aaa3f-cd92-4fb3-b83d-6c7b339447cf, 7, Finished, Available, Finished, False)

Gold orders rows: 4987
Gold orders columns: 9


SynapseWidget(Synapse.DataFrame, a4f5c2eb-d479-4a62-b7e4-5643e8b72f72)

In [6]:
print(
    "Null date_key:",
    df_gold_orders.filter(F.col("date_key").isNull()).count()
)

print(
    "Null country_key:",
    df_gold_orders.filter(F.col("country_key").isNull()).count()
)

print(
    "Null product_key:",
    df_gold_orders.filter(F.col("product_key").isNull()).count()
)

print(
    "Null page_key:",
    df_gold_orders.filter(F.col("page_key").isNull()).count()
)

print(
    "Null keyword_key:",
    df_gold_orders.filter(F.col("keyword_key").isNull()).count()
)

StatementMeta(, c28aaa3f-cd92-4fb3-b83d-6c7b339447cf, 8, Finished, Available, Finished, False)

Null date_key: 0
Null country_key: 0
Null product_key: 0
Null page_key: 0
Null keyword_key: 0


In [7]:
(
    df_gold_orders.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("fact_seo_orders")
)

print("Gold fact_seo_orders written successfully")

StatementMeta(, c28aaa3f-cd92-4fb3-b83d-6c7b339447cf, 9, Finished, Available, Finished, False)

Gold fact_seo_orders written successfully


In [8]:
df_gold_orders_check = spark.read.table(
    "fact_seo_orders"
)

print(
    "Gold fact_seo_orders rows:",
    df_gold_orders_check.count()
)

print(
    "Gold fact_seo_orders columns:",
    len(df_gold_orders_check.columns)
)

print(
    "Duplicate order IDs:",
    df_gold_orders_check
        .groupBy("order_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

display(df_gold_orders_check.limit(10))

StatementMeta(, c28aaa3f-cd92-4fb3-b83d-6c7b339447cf, 10, Finished, Available, Finished, False)

Gold fact_seo_orders rows: 4987
Gold fact_seo_orders columns: 9
Duplicate order IDs: 0


SynapseWidget(Synapse.DataFrame, 6df92538-b248-4d25-aceb-8d627dee4acd)